In [1]:
from openai import OpenAI

from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback
from toyaikit.chat import IPythonChatInterface

import os

from dotenv import load_dotenv
load_dotenv()


True

In [2]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [3]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [4]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [5]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [6]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.deepseek.com",
    timeout=300
)

# Define the model to use
llm_client = OpenAIChatCompletionsClient(
    model='deepseek-v4-flash',
    client=openai_client
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [7]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Command,Description
ollama list,List downloaded models
ollama pull <model>,Download a model
ollama run <model>,Start an interactive chat
ollama rm <model>,Remove a model
ollama serve,Start the Ollama API server


In [8]:
result.cost

CostInfo(input_cost=Decimal('0.00040614'), output_cost=Decimal('0.00021364'), total_cost=Decimal('0.00061978'))

In [9]:
result.all_messages

[{'role': 'system',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'How do I run Olama locally?'},
 ChatCompletionMessage(content='Let me look up information on running Olama locally!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_9KYpp93YA1uQ96Wmc6XX2579', function=Function(arguments='{"query": "Olama run locally"}', name='search'), type='function', index=0)], reasoning_content='Let me search for information about running Olama loca

In [10]:
runner.run()

You: Hi


-> Response received


You: Can I get the Cert?


-> Response received


-> Response received


...

-> Response received


-> Response received


You: stop


Chat ended.


LoopResult(new_messages=[{'role': 'system', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."}, {'role': 'user', 'content': 'Hi'}, ChatCompletionMessage(content="Hello! 👋 Welcome! I'm the teaching assistant for this course. \n\nHow can I help you today? Do you have any questions about the course material, assignments, or anything else you'd like to discuss? Feel free to ask away!", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='The student hasn\'t asked a question yet, just said "Hi". Let me resp